# Chapter 8 — ML in Production
### ML Engineer (Production) Interview Playbook

**Topics:** Feature Store · Model Serving · Versioning · Monitoring · Data Drift · Concept Drift · A/B Testing

> This chapter is the heart of your role. This is exactly where a model in a notebook differs from a
> real ML system. In a notebook, the model's story ends with an AUC number; in production, the model
> must be served, monitored, versioned, diagnosed when it breaks, and replaced without downtime. The
> lead wants to see whether you understand the full model lifecycle and data problems at scale.
>
> **Interview tip:** The sentence to say in the interview: **"The model is the smallest part of an ML
> system."** Most of the work is data engineering, serving, monitoring, and infrastructure — exactly
> what the previous chapters built.

### The ML production lifecycle

```
Data → Features (Feature Store) → Training → Evaluation
  → Model Registry (versioning) → Deployment (shadow/canary)
  → Serving (online/batch) → Monitoring (drift/quality)
  → (trigger) Retraining → back to the start
```

## 8.1 Feature Store

A Feature Store is a centralized repository of features, and it solves two fundamental problems:
(1) sharing and reusing features across teams and models (instead of every team rebuilding the same
feature with slightly different logic), and (2) keeping training time and serving time consistent.

| Component | Use | Requirement |
|---|---|---|
| Offline Store | Building a training set from large historical data | High throughput (warehouse / S3) |
| Online Store | Fetching features for real-time inference | Very low latency (Redis / key-value) |

### Training-Serving Skew

The most common and most insidious bug in ML systems: features are computed differently during
training and during serving (e.g. computed with pandas in training but rewritten in serving code, with
a small difference in normalization or missing-value handling). Result: the model looks great in
testing but performs badly in production, and weeks are spent finding the cause.

**Note:** The root-cause fix: a single source of truth for feature-generation logic — the same
code/definition that builds the offline feature must also build the online one. This is exactly what a
feature store does.

### Point-in-Time Correctness (preventing data leakage)

When building a training set, for every sample you must only pull features that were available **at
that historical moment**. If you include a feature that was computed later, the model gets information
from the future (data leakage) and the offline metric looks falsely great, but collapses in production.

In [ ]:
# Example of data leakage in fraud detection:
# feature "number of complaints filed for this transaction"
# -> this feature only exists AFTER the fraud is discovered!
# The model learns to rely on it, but it's not available at inference time.

# Correct: a point-in-time join
# for a transaction at time T, only use features computed before T


### Q8.1 — What problem does a Feature Store solve?

Three problems. First, training-serving skew: without a feature store, feature-generation logic is
written separately in the training pipeline and the inference service, and their small differences
cause the model to degrade in production; a feature store provides a single definition for both paths.
Second, reuse: expensive features (like a 30-day rolling transaction average) are computed once and
shared across teams and models. Third, the different needs of offline and online: the offline store
gives large historical data for training, and the online store gives the same features with
sub-millisecond latency for real-time inference. On top of that, it guarantees point-in-time
correctness so data leakage doesn't happen when building a training set.

## 8.2 Model Serving

Serving a model means making it available for prediction. The business need determines the serving
pattern, not personal preference:

| Pattern | What it is | Example |
|---|---|---|
| Online / Real-time | Synchronous response to each request; low latency is critical | Fraud check on a transaction in the moment |
| Batch | Prediction over a large volume, scheduled | Nightly risk score for all users |
| Streaming | Continuous processing of an event stream | Behavioral monitoring over a transaction stream |

In [ ]:
# Serving with FastAPI -- the model is loaded once at startup

@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.model = joblib.load(settings.model_path)   # not on every request!
    yield

@app.post("/predict")
async def predict(req: PredictRequest):
    features = await feature_store.get_online(req.user_id)   # real-time features
    score = app.state.model.predict_proba([features])[0, 1]
    return {"score": float(score), "model_version": settings.model_version}


**Interview tip:** Always return `model_version` in the response and in the log. Without it, when
you later need to know which model a specific prediction came from (for debugging, auditing, or A/B
analysis), you have nothing to go on. In a financial environment this is mandatory for auditability.

### Latency optimization

- Load the model once at startup, not on every request.
- Precompute expensive features in the online store (not at request time).
- Batch requests (dynamic batching) for GPU efficiency.
- A lighter or compressed model (quantization, distillation, ONNX) if latency is critical.
- Cache predictions for repeated inputs.

Dedicated serving tools: TorchServe, TensorFlow Serving, Triton (GPU-optimized), BentoML, KServe (on
Kubernetes). For lightweight models (like decision trees in fraud detection), a plain FastAPI service
is often sufficient and simpler.

### Q8.2 — Your inference service's latency is 500ms, but it needs to be under 100ms. What do you do?

First I measure where the time goes — usually one of three things: feature fetching, the inference
itself, or network/serialization overhead. If feature fetching is expensive, I precompute features and
put them in a low-latency online store (Redis) instead of computing them on the fly. If the model
itself is slow, options include: a lighter model, compression (quantization/distillation), converting
to an optimized format (ONNX), or batching for efficiency. If the model is being loaded on every
request (a common mistake), I move that to startup. Caching predictions for repeated inputs also helps.
Finally, I make the trade-off explicit: often accuracy can be traded for a meaningful latency reduction
— the decision should be based on business value.

## 8.3 Model Versioning and Reproducibility

Reproducibility means being able to exactly reconstruct a model. For this, versioning code alone isn't
enough; four things must be versioned together:

- **Code —** the repository commit hash.
- **Data —** the dataset's version/hash (with a tool like DVC or a warehouse snapshot).
- **Config —** hyperparameters and training settings.
- **Environment —** library versions (this is where Docker comes in).

A Model Registry (like MLflow) attaches this metadata to every model artifact and manages the staging
lifecycle (staging → production → archived).

**Interview tip:** The most important practical benefit of versioning: fast rollback. If a new model
performs badly in production, you need to be able to revert to the previous version within minutes. If
"rolling back" requires retraining, you've failed.

### Q8.3 — How do you guarantee a model is reproducible?

By versioning every input to the process, not just the code: (1) the training code's commit, (2) the
exact version or hash of the dataset (because the same code on different data gives a different model),
(3) hyperparameters and the random seed, and (4) the runtime environment with exact library versions —
pinned via a Docker image. I record all of this as metadata in a model registry (like MLflow) alongside
the model artifact, together with evaluation metrics. This way I can trace every production model back
to its exact data and code — which is also required for auditability in a financial environment.

## 8.4 Monitoring

Monitoring an ML system has two layers, and both are essential. A common mistake: only the first layer
is monitored, and the model breaks silently.

| Layer | What's monitored |
|---|---|
| Operational (like any service) | Latency (p50/p95/p99), throughput, error rate, CPU/memory usage |
| Model and data | Input feature distribution, output distribution, missing-value rate, drift, prediction quality |

**Note:** ML's unique challenge: a model breaks **silently**. A normal service that breaks throws
errors and fires alerts. A broken model keeps confidently returning numbers — just wrong ones. That's
why monitoring input/output distributions is critical, not just service health monitoring.

### The label delay problem

In fraud detection, the true label ("was this transaction actually fraud?") might only become known
weeks or months later (after a user complaint or investigation). This means you can't measure the
model's real quality in real time. Solution: rely on early (proxy) signals:

- Drift in the input feature distribution.
- Shifts in the model's output distribution (e.g. the high-score rate suddenly doubles).
- The manual-review rate and its outcomes (faster feedback from the review team).
- Intermediate business metrics (block rate, customer complaint rate).

### Q8.4 — What do you monitor in a production model?

Two layers. The operational layer, like any other service: latency (especially p95/p99), throughput,
error rate, resource usage. The ML-specific layer: the input feature distribution (to catch data
drift), the model's output distribution (e.g. if the average risk score suddenly jumps, that's a sign
of a problem), the missing/invalid value rate in the input (which is often a sign of an upstream
pipeline breaking), and prediction quality whenever a true label arrives. The important point is that
in many domains like fraud, labels arrive with delay, so you can't just wait for accuracy to drop; you
have to watch early signals like drift and intermediate business metrics. I also keep `model_version`
in every log so I can attribute a problem to a specific version.

## 8.5 Data Drift and Concept Drift

This distinction is one of the most frequently asked questions in ML production interviews. The key to
understanding it is looking at the probabilities involved:

| Type | What changes | Example in fraud |
|---|---|---|
| Data Drift (Covariate Shift) | The input distribution P(X); the X→y relationship stays fixed | A wave of users arrives from a new country |
| Concept Drift | The relationship itself, P(y\|X) | A new fraud pattern that didn't exist before |
| Label Shift | The output distribution P(y) | The overall fraud rate suddenly rises |

### Detecting drift

- **Data drift:** statistical tests on the input distribution — the KS test (for continuous features),
  Chi-square (for categorical), or PSI (Population Stability Index), common in the financial industry.
- **Concept drift:** only detectable through an actual drop in real metrics (once labels arrive) —
  because the X→y relationship has changed, not the input. That's why it's harder and slower to detect.

**Interview tip:** A subtle point that scores well: data drift doesn't necessarily lead to performance
degradation (if the model also generalizes well to the new region). But concept drift almost always
degrades performance. So don't blindly translate every drift alert into "retrain"; measure its actual
impact first.

### Q8.5 — What's the difference between data drift and concept drift, and how do you respond to each?

Data drift means the input distribution P(X) has changed but the relationship between input and output
stays fixed — e.g. a wave of users from a new country has arrived. Concept drift means the relationship
itself, P(y|X), has changed — e.g. fraudsters have invented a new method that the old pattern no longer
captures. Detection: I catch data drift with statistical tests on the input feature distribution
(KS-test, PSI), which is fast and needs no labels; concept drift can only be seen through an actual
metric drop once labels arrive. The response to both is usually retraining on new data, but the
important nuance: data drift doesn't always lead to a performance drop, so I measure its real impact
first before deciding to retrain. If the drift is caused by a pipeline bug (not a real change in the
world), retraining is the wrong move and the bug needs to be fixed instead.

## 8.6 Deployment and A/B Testing

A new model should never take 100% of traffic overnight. Gradual rollout controls risk:

| Strategy | How it works | What it measures |
|---|---|---|
| Shadow | The new model predicts, but its output isn't used | Technical stability and output difference, with zero user risk |
| Canary | A small traffic percentage (e.g. 5%) goes to the new model | Real behavior with limited risk; gradually increased |
| A/B Test | Traffic is randomly split between control and treatment | Causal effect on a business metric with statistical significance |
| Blue-Green | Two complete environments; a one-shot switch with instant rollback | Zero-downtime deployment with fast rollback |

### Doing A/B testing correctly

- A random, stable split (the same user always stays in the same group).
- Define the primary metric before starting, not after seeing the result.
- Compute the sample size and duration needed for statistical significance.
- Avoid peeking (repeatedly checking and stopping early once you see the result you want), which
  inflates the false-positive rate.

**Note:** Metric warning — the most important point of this chapter: improving offline AUC doesn't
necessarily lead to improving the business. In fraud, a model with higher AUC might produce more false
positives and block real users; the cost of that (dissatisfaction, customer churn) can outweigh the
benefit of catching a few more frauds. Always validate against the real business metric.

### Q8.6 — How do you confidently bring a new model to production?

Gradually, with the ability to roll back at every stage. First, shadow deployment: the new model
predicts on real traffic but its output isn't used — this lets me measure technical stability, latency,
and output difference from the current model with zero user risk. Then canary: I send a small
percentage of traffic (e.g. 5%) to the new model and watch operational and business metrics, gradually
increasing it if it's healthy. Then an A/B test with a random split and a pre-defined primary metric, to
measure the causal effect on the business metric with statistical significance — not just offline
accuracy. Throughout all stages I keep fast rollback to the previous version available (thanks to the
model registry) and active alerting.

## 8.7 Retraining

Models decay over time (model decay). The question isn't "should we retrain?" but "when and how?"

- **Scheduled:** e.g. weekly; simple and predictable, but might be earlier or later than actually
  needed.
- **Performance-based trigger:** when a metric drops below a threshold; more precise but requires
  labels (which might arrive late).
- **Drift-based trigger:** when significant drift is detected; reacts faster and doesn't need labels.

**Note:** A dangerous trap: the feedback loop. If the model blocks certain transactions, you never see
their real outcome; so the next training data becomes biased, and the model gets reinforced in the same
direction. A common fix: keep a small control group (some traffic the model isn't applied to, or is
applied differently) so unbiased data keeps being collected.

Golden rule: automated retraining must go through the same quality gates as a manual deployment —
evaluation on a holdout set, comparison against the baseline, and gradual rollout. Automated retraining
without gates is a fast way to ship a bad model to production.

### Q8.7 — How often do you retrain a model?

The correct answer is "it depends," and you should explain the criterion. The deciding factor is how
fast the real world changes (the rate of concept drift): in fraud, where fraudsters actively change
their behavior, more frequent retraining (e.g. weekly or even daily) makes sense; in a more stable
domain, monthly is enough. My preference is a combination: a scheduled retraining as a baseline, plus a
drift- or metric-drop-based trigger for faster reaction to sudden changes. It's important that automated
retraining goes through the same quality gates (evaluation on a holdout and comparison with the current
model) and that the new model is deployed gradually (shadow/canary) — otherwise you might automatically
ship a worse model to production.

## Quick Chapter Summary (Cheat Sheet)

Review this on interview day:

- **Feature Store:** offline (training) + online (inference); solves training-serving skew with a
  single feature definition; point-in-time correctness to prevent data leakage.
- **Serving:** online/batch/streaming; load the model at startup; put `model_version` in the response
  and logs; cut latency via feature precomputation, batching, and compression.
- **Versioning:** code + data + config + environment; a model registry; the main goal is fast rollback.
- **Monitoring:** two layers (operational + model); a model breaks silently; label delay → rely on
  early signals.
- **Drift:** data drift = P(X) change (via KS/PSI); concept drift = P(y|X) change (only via a metric
  drop); data drift doesn't always cause performance degradation.
- **Deployment:** shadow → canary → A/B; define the primary metric up front; no peeking; offline AUC ≠
  business value.
- **Retraining:** schedule + drift/performance trigger; watch for the feedback loop (keep a control
  group); automated retraining needs quality gates.